In [0]:
# Import packages and functions needed 
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType


# Function that identifies and explodes array and struct fields
def expand_arrays(df):
    array_columns = []
    
    # search and select the array columns
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # loop the array columns
    for col_name in array_columns:
        # Explode each array column
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verify whether the data type of the array is a struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # If it does, extract the struct subfields, create new fields and delete the original one
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # if the data type is not StructType (ie, StringType o DoubleType), replace the original column with the exploded one
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # delete the temporal exploded field
        df = df.drop(f"exploded_{col_name}")
    
    return df


# Función para separar el campo usando '~' y ',' como delimitadores
def split_multiple_delimiters(df, input_col, output_col):
    # Usamos regexp_replace para normalizar los delimitadores a uno solo (por ejemplo ',')
    normalized_col = F.regexp_replace(input_col, '[~,]+', ',')
    # Hacemos split del resultado normalizado
    return df.withColumn(output_col, F.split(normalized_col, ','))


# Definir la función para capitalizar la primera letra, limpiar espacios y reemplazar '_'
def clean_text(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.concat(
            F.upper(F.substring(F.trim(F.regexp_replace(F.col(input_col), '_', ' ')), 1, 1)),
            F.lower(F.substring(F.trim(F.regexp_replace(F.col(input_col), '_', ' ')), 2, 1000))
        )
    )

# funcion para extraer el contenido entre corchetes
def extract_string_content(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.when(
            F.col(input_col).rlike(r'\[.*?\]'),  # Si contiene corchetes
            F.regexp_replace(  # Remover las comillas dobles después de extraer el contenido
                F.regexp_extract(F.col(input_col), r'\[(.*?)\]', 1),
                r'"', ''  # Reemplazar todas las comillas dobles por un string vacío
            )
        ).otherwise(F.col(input_col))  # Si no tiene corchetes, dejar el valor original
    )


In [0]:
# Import tables

## bmdm table
gdm = spark.table("crm_reporting.dim_gdm_brand_profile")
# print(gdm.count()) #10496032
# print(len(gdm.columns))


####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE
df = gdm
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 78

# select the founded fields
gdm_cols_filt = df.select(non_completely_null_columns)#.\
  # withColumnRenamed("created_dt",'created_date')



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store','category','product_category']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in gdm_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
# print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
gdm_cols_filt = gdm_cols_filt.select(columns_wt_dt)
gdm_cols_filt.createOrReplaceTempView("gdm_cols_filt_vw")

In [0]:
### Merged tables
merged_gdm = spark.sql("""
select 
  upper(a.source_name) as source_name,
  lower(a.registration_sub_source) AS registration_sub_source,
  a.source_customer_id,
  -- e.channel_name, 
  --c.brand_customer_id,
  --a.last_modified_dt as created_date,
  case when a.created_dt is null then date(a.last_modified_dt)
    else date(a.created_dt) end as created_date,
  d.*
from prod_latam_catalog.crm_reporting.dim_customer a
inner join prod_latam_catalog.crm_reporting.dim_customer_bridge c
  on a.source_customer_id = c.source_customer_id 
  and a.brand_code = c.brand_code 
  and a.brand_country = c.brand_country
-- inner join expld_gdm_vw d 
inner join gdm_cols_filt_vw d
  on c.brand_mdm_id = d.brand_mdm_id 
  and c.brand_code = d.brand_code 
  and c.brand_country = d.brand_country
-- left join prod_latam_catalog.crm_reporting.dim_channel e 
--   on concat(UPPER(a.source_name),"_",UPPER(a.acq_source),"_",UPPER(a.registration_source)) = e.row_key
where upper(a.source_name) IN ('DEMANDWARE','JEBBIT','SITECORE') 
  AND (lower(a.registration_source) LIKE ('%quiz%') 
     OR lower(a.registration_source) LIKE ('%diagnos%')
     OR lower(a.registration_source) LIKE ('%website%'))
  AND lower(a.registration_sub_source) NOT IN ('cart checkout','header')
  -- lower(a.registration_sub_source) not in ('contest',null,'registration','','Footer')
  --AND to_date(a.created_dt) >= '2023-01-01'
group by all
order by registration_sub_source,brand_code, brand_country

""")

# print(merged_gdm.count()) #303918

In [0]:
# call the expand array function twice
expld_gdm_1 = expand_arrays(merged_gdm) # explode first array levels
expld_gdm_1 = expand_arrays(expld_gdm_1) # explode second array levels

# print(expld_gdm.count()) # 433098
# print(len(expld_gdm.columns)) ## 81
# display(expld_gdm.limit(1))
expld_gdm_1.createOrReplaceTempView("expld_gdm_1_vw")
  # print(expld_gdm_1.count()) # 433098

In [0]:
expld_gdm = spark.sql("""
with dim_customer_1 as (
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    case when created_dt is null then date(last_modified_dt)
      else date(created_dt) end as created_date,
    concat(UPPER(source_name),"_",UPPER(acq_source),"_",UPPER(registration_source)) as row_key
  from prod_latam_catalog.crm_reporting.dim_customer
),

dim_customer_date as	(
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(created_date) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_date asc) AS created_date
  from dim_customer_1
),

dim_customer_acq_channel as	(
  select distinct
    brand_country,
    brand_code,
    source_customer_id,
    FIRST_VALUE(row_key) 
      OVER (PARTITION BY brand_country,brand_code,source_customer_id ORDER BY created_date asc) AS row_key
  from dim_customer_1
),

dim_channel as (
  select distinct
    row_key,
    channel_name
  from prod_latam_catalog.crm_reporting.dim_channel
)

select
  a.*,
  e.channel_name,
  case when b.created_date is null then null
      else 'New' end as new_client_key,
  d.row_key
from expld_gdm_1_vw a
left join dim_customer_date b
  on a.brand_country = b.brand_country
  and a.brand_code = b.brand_code 
  and a.source_customer_id = b.source_customer_id 
  and a.created_date = b.created_date
left join dim_customer_acq_channel d
  on a.brand_country = d.brand_country
  and a.brand_code = d.brand_code 
  and a.source_customer_id = d.source_customer_id 
left join dim_channel e 
  on d.row_key = e.row_key
""")

expld_gdm.createOrReplaceTempView("expld_gdm_vw")
# print(expld_gdm.count()) #301670

In [0]:
%sql
select distinct 
brand_country,brand_code,
source_name,acq_source,registration_source,
concat(UPPER(source_name),"_",UPPER(acq_source),"_",UPPER(registration_source)) as row_key,
registration_sub_source,
date(created_dt) as created_dt
from prod_latam_catalog.crm_reporting.dim_customer
-- where source_customer_id = 'd97023081c8689204c82cfd4c268f2df' -- channel null
where source_customer_id = '996c464664095a4388dd6dab77d25998' -- channel dsf service
  -- and source_name ='DEMANDWARE'
--LOWER(acq_sub_source)='newsletter' 'hola'
order by brand_country,brand_code,created_dt


brand_country,brand_code,source_name,acq_source,registration_source,row_key,registration_sub_source,created_dt
MEX,LAN,DEMANDWARE,Website,Quizzes/Diagnostics Tool,DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTICS TOOL,Product Finder,null


In [0]:
%sql
select
  row_key,
  channel_name,
  count(distinct brand_mdm_id) counts
from expld_gdm_vw
group by all
order by row_key

row_key,channel_name,counts
null,null,652
DEMANDWARE_WEBSITE_DIAGNOSTIC TOOL,E-commerce,2883
DEMANDWARE_WEBSITE_DIAGNOSTIC TOOLS,E-commerce,1
DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTIC TOOLS,DSF Services,10587
DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTICS TOOL,DSF Services,57042
DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTICS TOOLS,DSF Services,19517
DEMANDWARE__,E-commerce,3
JEBBIT_FACEBOOK_QUIZZES/DIAGNOSTICS TOOLS,Facebook,115539
JEBBIT_FU LL PAGE_QUIZZES/DIAGNOSTICS TOOLS,Jebbit,90
JEBBIT_FU LL PAG_QUIZZES/DIAGNOSTICS TOOLS,Jebbit,11


In [0]:
# %sql
#   select brand_code,brand_country,source_customer_id
#   from prod_latam_catalog.crm_reporting.dim_customer
#   where concat(UPPER(source_name),"_",UPPER(acq_source),"_",UPPER(registration_source)) = 'DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTICS TOOL'

In [0]:
%sql

with tmp as (
  select brand_code,brand_country,source_customer_id
  from prod_latam_catalog.crm_reporting.dim_customer
  where concat(UPPER(source_name),"_",UPPER(acq_source),"_",UPPER(registration_source)) = 'DEMANDWARE_WEBSITE_QUIZZES/DIAGNOSTICS TOOL'
) 

select count(1)
from tmp

count(1)
70278


In [0]:
%sql
select distinct 
source_name,
registration_source,
registration_sub_source,
acq_source
from prod_latam_catalog.crm_reporting.dim_customer
where source_name ='DEMANDWARE'
--LOWER(acq_sub_source)='newsletter'
 

source_name,registration_source,registration_sub_source,acq_source
DEMANDWARE,Registration,Contact us,Website
DEMANDWARE,null,null,Website
DEMANDWARE,None,Cart Checkout,Website
DEMANDWARE,Newsletter,registration,Website
DEMANDWARE,Account Creation,Cart Checkout,Website
DEMANDWARE,Quizzes/Diagnostics Tool,Product Finder,Website
DEMANDWARE,null,registration,Website
DEMANDWARE,Registration,null,Website
DEMANDWARE,newsletter.account.create,null,Website
DEMANDWARE,Newsletter,Header,Website


In [0]:
%sql
select distinct 
  a.brand_country,a.brand_code,b.channel_name,a.source_customer_id,
  date(a.created_dt) as created_dt,date(a.last_modified_dt) as last_modified_dt,
  a.source_name,b.brand_mdm_id
from prod_latam_catalog.crm_reporting.dim_customer a 
inner join expld_gdm_vw b
  on a.source_customer_id = b.source_customer_id
where a.source_customer_id in ('0001cf78833f0d927843e0ef7e8fdc21', -- new)
                               '0007f8a4ffbba81ff709ef829283bd2d') --not new
order by source_customer_id,created_dt asc
--limit 1000

brand_country,brand_code,channel_name,source_customer_id,created_dt,last_modified_dt,source_name,brand_mdm_id
CHI,KIE,DSF Services,0001cf78833f0d927843e0ef7e8fdc21,2022-06-27,2022-06-27,SFMC_FACEBOOK_LEADAD,ff0be336fafc912534aadcdad9452b69
CHI,LRP,DSF Services,0001cf78833f0d927843e0ef7e8fdc21,2023-11-22,2023-11-22,SFMC_FACEBOOK_LEADAD,ff0be336fafc912534aadcdad9452b69
CHI,KER,DSF Services,0001cf78833f0d927843e0ef7e8fdc21,2023-12-11,2023-12-11,DEMANDWARE,ff0be336fafc912534aadcdad9452b69
CHI,VIC,DSF Services,0001cf78833f0d927843e0ef7e8fdc21,2024-05-25,2024-05-25,SFMC_FACEBOOK_LEADAD,ff0be336fafc912534aadcdad9452b69
CHI,KER,DSF Services,0001cf78833f0d927843e0ef7e8fdc21,2024-06-30,2024-06-30,SFMC_FACEBOOK_LEADAD,ff0be336fafc912534aadcdad9452b69
ARG,VIC,null,0007f8a4ffbba81ff709ef829283bd2d,2020-10-21,2022-02-24,SFMC_LANDINGPAGE_LOREAL,d38359d7201e4a1776b10b45ef3d9996
ARG,RLP,null,0007f8a4ffbba81ff709ef829283bd2d,2023-07-01,2023-07-01,SFMC_E-RETAILER_PARFUMERIE,d38359d7201e4a1776b10b45ef3d9996
ARG,KER,null,0007f8a4ffbba81ff709ef829283bd2d,2023-07-09,2023-07-09,SFMC_FACEBOOK_LEADAD,d38359d7201e4a1776b10b45ef3d9996
ARG,KER,null,0007f8a4ffbba81ff709ef829283bd2d,2023-09-18,2023-09-18,SITECORE,d38359d7201e4a1776b10b45ef3d9996
ARG,LOP,null,0007f8a4ffbba81ff709ef829283bd2d,2024-03-07,2024-03-07,SFMC_FACEBOOK_LEADAD,d38359d7201e4a1776b10b45ef3d9996


In [0]:
# %sql
# select *
# from expld_gdm_vw
# where source_customer_id in ('0001cf78833f0d927843e0ef7e8fdc21', -- new)
#                              '0007f8a4ffbba81ff709ef829283bd2d') --not new

## UNITARIAS

In [0]:
%sql
select *
  --brand_code,brand_country,source_customer_id,brand_customer_id, source_name
-- from prod_latam_catalog.crm_reporting.dim_customer_bridge
from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
-- where brand_customer_id = '0b84f682c3a17663c07a2d00c9446f96' 
where source_customer_id = '65e333b480f53ede74342c3c3600231d'

bridge_customer_id,mdm_source,source_name,source_customer_id,customer_id,brand_customer_id,global_customer_id,brand_mdm_id,global_mdm_id,record_status,sys_last_modified_by,sys_last_modified_dt,etl_batch_id,gender,year_of_birth,preferred_lang,brand_code,brand_country
368016b3b53e7bb6d5381b5648fd3cbb,DDM,SFMC_E-RETAILER_JULERIAQUE,65e333b480f53ede74342c3c3600231d,2d993c8675e7b480443ed495233284c0,c4cfaa9035148a911e96f3200173354f,c144f327afc3925a5cf719c09a410427,dc9a933bc876c0482aa33b880174e38f,319bfef5f73cb56e3f799707b1672ab4,ACTIVE,spark_user,2024-12-04T08:34:15.62Z,1081_20241204080533,UNKNOWN,null,ES,RLP,ARG
942638fd56848015917413f6903a6954,DDM,SFMC_E-RETAILER_JULERIAQUE,65e333b480f53ede74342c3c3600231d,0260013ad3f1f0df9b441c2848b460fc,6f8df1ab204b47b37d228af74770145e,c144f327afc3925a5cf719c09a410427,1203191c4faf6aa710ee93d1e1e16709,319bfef5f73cb56e3f799707b1672ab4,ACTIVE,spark_user,2024-12-04T08:34:15.62Z,1081_20241204080533,UNKNOWN,null,ES,LAN,ARG
b9f98a0cd85dacfb7ac22259e7f50356,DDM,SFMC_LANDINGPAGE_LOREAL,65e333b480f53ede74342c3c3600231d,f52c0501392efec7557e5e070a3ba83f,08e986fdd3d458442d8922707296d99c,dece83177295686e14f9d2d93f94518f,a673824b7b8ed3f788d901ad9cffdd6d,13cc02e54d78e60fbe7817e1dc57f834,ACTIVE,spark_user,2024-12-04T08:34:15.62Z,1081_20241204080533,FEMALE,1977,ES,LAN,ARG


In [0]:
%sql
select --*
  brand_code,brand_country,source_customer_id,brand_customer_id,source_name
from prod_latam_catalog.crm_reporting.dim_customer_bridge
-- from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
-- where brand_customer_id = '0b84f682c3a17663c07a2d00c9446f96' 
where source_customer_id = '65e333b480f53ede74342c3c3600231d'

brand_code,brand_country,source_customer_id,brand_customer_id,source_name
LAN,ARG,65e333b480f53ede74342c3c3600231d,08e986fdd3d458442d8922707296d99c,SFMC_LANDINGPAGE_LOREAL
LAN,ARG,65e333b480f53ede74342c3c3600231d,6f8df1ab204b47b37d228af74770145e,SFMC_E-RETAILER_JULERIAQUE
RLP,ARG,65e333b480f53ede74342c3c3600231d,c4cfaa9035148a911e96f3200173354f,SFMC_E-RETAILER_JULERIAQUE


In [0]:
%sql
select distinct 
  brand_country,brand_code,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt,source_name
from prod_latam_catalog.crm_reporting.dim_customer_hist
where source_customer_id = '65e333b480f53ede74342c3c3600231d'
  and brand_country = 'ARG'
  and brand_code = 'LAN'
order by last_modified_dt

brand_country,brand_code,source_customer_id,created_dt,last_modified_dt,source_name
ARG,LAN,65e333b480f53ede74342c3c3600231d,2024-06-01,2024-06-01,SFMC_LANDINGPAGE_LOREAL
ARG,LAN,65e333b480f53ede74342c3c3600231d,2024-03-01,2024-03-01,SFMC_E-RETAILER_JULERIAQUE


In [0]:
%sql
select distinct 
  brand_country,brand_code,source_customer_id,date(created_dt) as created_dt,date(last_modified_dt) as last_modified_dt,source_name
from prod_latam_catalog.crm_reporting.dim_customer
where source_customer_id = '65e333b480f53ede74342c3c3600231d'
  and brand_country = 'ARG'
  and brand_code = 'LAN'
order by last_modified_dt

brand_country,brand_code,source_customer_id,created_dt,last_modified_dt,source_name
ARG,LAN,65e333b480f53ede74342c3c3600231d,2024-06-01,2024-06-01,SFMC_LANDINGPAGE_LOREAL
ARG,LAN,65e333b480f53ede74342c3c3600231d,2024-03-01,2024-03-01,SFMC_E-RETAILER_JULERIAQUE


In [0]:
%sql
select source_customer_id,brand_country,brand_code,interaction_sub_type,date(interaction_dt) as interaction_dt,source_name
from prod_latam_catalog.crm_reporting.fact_customer_interactions
-- WHERE source_customer_id in ('6eddb4ab90a32d1bd5473ea62bda819a','589496032c6b74b0fe9bf374dd897338') 
where interaction_sub_type = 'Profile Created' 
  and source_customer_id = '65e333b480f53ede74342c3c3600231d'
  and brand_country = 'ARG'
  and brand_code = 'LAN'
  -- and brand_country = 'MEX'  
  -- and brand_code = 'LAN'

source_customer_id,brand_country,brand_code,interaction_sub_type,interaction_dt,source_name
65e333b480f53ede74342c3c3600231d,ARG,LAN,Profile Created,2024-06-01,SFMC_LANDINGPAGE_LOREAL
65e333b480f53ede74342c3c3600231d,ARG,LAN,Profile Created,2024-03-01,SFMC_E-RETAILER_JULERIAQUE


In [0]:
%sql
select distinct snapshot_date_key,brand_country,brand_code,brand_customer_id,lifetime_new_consumer_activity_segment_key
from crm_reporting.fact_segment_by_brand
where brand_customer_id = '6f8df1ab204b47b37d228af74770145e' 
  --and snapshot_date_key = '20240630'
order by snapshot_date_key

snapshot_date_key,brand_country,brand_code,brand_customer_id,lifetime_new_consumer_activity_segment_key
20240331,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_1C
20240430,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_2C
20240531,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_3C
20240630,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_46C
20240731,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_46C
20240831,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_79C
20240930,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_79C
20241031,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_79C
20241130,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_1012C
20241208,ARG,LAN,6f8df1ab204b47b37d228af74770145e,DDM_LNCAS_1012C


In [0]:
# %sql
# select *--brand_code,brand_country,source_customer_id,brand_customer_id
# from prod_latam_catalog.crm_reporting.dim_customer_bridge
# -- from prod_latam_catalog.crm_reporting.etl_stg_customer_bridge
# where source_customer_id = '0001cf78833f0d927843e0ef7e8fdc21'